# Notebook 15: External Transfer Validation

This notebook writes the external transfer summary artifact and records blocked status when external datasets are not available in the current repository snapshot.

In [3]:
from pathlib import Path
import math
from typing import Iterable, Optional

import numpy as np
import pandas as pd

try:
    import joblib
except Exception:
    joblib = None

try:
    from sklearn.metrics import mean_squared_error, r2_score
except Exception as exc:
    raise RuntimeError("scikit-learn is required for NB15 external transfer scoring") from exc

NOTEBOOK_ROOT = Path.cwd() if Path.cwd().name == "15_external_transfer_no_scripts" else Path(r"c:/Users/obalo/Downloads/PhDOneDrive/PhD_Research_Operating_System/github_org_bootstrap/phd-geothermal-ml/manual_bootstrap/step_by_step_notebooks/15_external_transfer_no_scripts")
PROJECT_ROOT = NOTEBOOK_ROOT.parents[2]
PHD_OS_ROOT = PROJECT_ROOT.parents[1]
INPUT_ROOT = NOTEBOOK_ROOT / "inputs"
OUTPUT_ROOT = NOTEBOOK_ROOT / "outputs" / "summary"
TABLE_ROOT = OUTPUT_ROOT / "tables"
REPORT_ROOT = OUTPUT_ROOT / "reports"
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_COL = "Temperature(C)"
GROUP_COL = "State"
DOMAIN_HINTS = {
    "turkey": "turkey_western_anatolia",
    "anatolia": "turkey_western_anatolia",
    "morocco": "morocco_northern",
    "waikato": "new_zealand_waikato",
    "zealand": "new_zealand_waikato",
    "new_zealand": "new_zealand_waikato",
}

EXTERNAL_DATA_PATTERNS = [
    "**/*external*.csv",
    "**/*transfer*.csv",
    "**/*turk*.csv",
    "**/*morocco*.csv",
    "**/*waikato*.csv",
    "**/*zealand*.csv",
    "**/*external*.parquet",
    "**/*transfer*.parquet",
    "**/*external*.xlsx",
    "**/*transfer*.xlsx",
]

MODEL_PATTERNS = [
    "**/*gradient_boosting*.joblib",
    "**/*gradient_boosting*.pkl",
    "**/*gradient_boosting*.pickle",
    "**/*nb16*.joblib",
    "**/*nb16*.pkl",
    "**/*frozen*.joblib",
    "**/*frozen*.pkl",
]

IGNORE_FILENAMES = {
    "external_transfer_summary.csv",
    "external_transfer_blocker_notes.md",
}


def _walk_patterns(root: Path, patterns: Iterable[str]) -> list[Path]:
    hits: list[Path] = []
    for pattern in patterns:
        hits.extend(root.glob(pattern))
    unique: list[Path] = []
    seen = set()
    for p in hits:
        if not p.is_file():
            continue
        if p.name in IGNORE_FILENAMES:
            continue
        if "outputs" in p.parts and "summary" in p.parts:
            continue
        if p not in seen:
            seen.add(p)
            unique.append(p)
    return unique


def discover_external_sources() -> list[Path]:
    roots = [INPUT_ROOT, PROJECT_ROOT, PHD_OS_ROOT]
    hits: list[Path] = []
    for root in roots:
        hits.extend(_walk_patterns(root, EXTERNAL_DATA_PATTERNS))
    unique: list[Path] = []
    seen = set()
    for p in hits:
        if p not in seen:
            seen.add(p)
            unique.append(p)
    return unique


def discover_model_artifacts() -> list[Path]:
    roots = [PROJECT_ROOT, PHD_OS_ROOT]
    hits: list[Path] = []
    for root in roots:
        hits.extend(_walk_patterns(root, MODEL_PATTERNS))
    unique: list[Path] = []
    seen = set()
    for p in hits:
        if p not in seen:
            seen.add(p)
            unique.append(p)
    return unique


def infer_domain_name(path: Path) -> str:
    text = " ".join([path.name, str(path.parent)]).lower()
    for key, domain in DOMAIN_HINTS.items():
        if key in text:
            return domain
    return path.stem.replace(" ", "_").replace("-", "_").lower()


def load_tabular(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx", ".xlsm", ".xls"}:
        return pd.read_excel(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")


def canonicalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    renamed = {}
    for col in df.columns:
        key = str(col).strip().lower().replace(" ", "_").replace("(", "").replace(")", "")
        if key in {"temperaturec", "temperature_c", "temp", "temperature"}:
            renamed[col] = TARGET_COL
        elif key in {"state", "province", "region", "domain"}:
            renamed[col] = GROUP_COL
    return df.rename(columns=renamed)


def select_features_for_model(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = [c for c in [TARGET_COL] if c in df.columns]
    if GROUP_COL in df.columns:
        drop_cols.append(GROUP_COL)
    return df.drop(columns=drop_cols, errors="ignore")


def choose_model_artifact() -> Optional[Path]:
    hits = discover_model_artifacts()
    if not hits:
        return None
    preferred = [p for p in hits if "gradient_boosting" in p.name.lower() or "nb16" in p.name.lower()]
    return preferred[0] if preferred else hits[0]


def score_external_dataset(df: pd.DataFrame, model) -> tuple[float, float, int, dict]:
    df = canonicalize_columns(df)
    if TARGET_COL not in df.columns:
        raise ValueError(f"Missing target column: {TARGET_COL}")
    if GROUP_COL not in df.columns:
        df[GROUP_COL] = "external_domain"

    y_true = pd.to_numeric(df[TARGET_COL], errors="coerce")
    valid = y_true.notna()
    df = df.loc[valid].copy()
    y_true = y_true.loc[valid].astype(float)

    X = select_features_for_model(df)
    if hasattr(model, "predict"):
        y_pred = model.predict(X)
    elif callable(model):
        y_pred = model(X)
    else:
        raise TypeError("Loaded model object does not support prediction")

    y_pred = pd.Series(np.asarray(y_pred).reshape(-1), index=y_true.index)
    rmse = float(math.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    meta = {
        "n_test": int(len(y_true)),
        "target_min": float(y_true.min()),
        "target_max": float(y_true.max()),
    }
    return rmse, r2, len(y_true), meta


def build_status_rows() -> pd.DataFrame:
    data_files = discover_external_sources()
    model_path = choose_model_artifact()
    status_rows = []

    if not data_files:
        for domain in ["turkey_western_anatolia", "morocco_northern", "new_zealand_waikato"]:
            status_rows.append({
                "domain": domain,
                "model_name": "gradient_boosting",
                "rmse": "N/A",
                "r2": "N/A",
                "n_test": "N/A",
                "status": "blocked",
                "notes": "external dataset not found in current repository snapshot",
            })
        return pd.DataFrame(status_rows)

    if model_path is None:
        for path in data_files:
            domain = infer_domain_name(path)
            status_rows.append({
                "domain": domain,
                "model_name": "gradient_boosting",
                "rmse": "N/A",
                "r2": "N/A",
                "n_test": "N/A",
                "status": "blocked",
                "notes": f"dataset found at {path}; model artifact not found",
            })
        return pd.DataFrame(status_rows)

    if joblib is None:
        for path in data_files:
            domain = infer_domain_name(path)
            status_rows.append({
                "domain": domain,
                "model_name": model_path.stem,
                "rmse": "N/A",
                "r2": "N/A",
                "n_test": "N/A",
                "status": "blocked",
                "notes": "joblib unavailable for model loading",
            })
        return pd.DataFrame(status_rows)

    model = joblib.load(model_path)
    for path in data_files:
        domain = infer_domain_name(path)
        try:
            df = load_tabular(path)
            rmse, r2, n_test, meta = score_external_dataset(df, model)
            status_rows.append({
                "domain": domain,
                "model_name": model_path.stem,
                "rmse": rmse,
                "r2": r2,
                "n_test": n_test,
                "status": "scored",
                "notes": f"target_range={meta['target_min']:.2f}..{meta['target_max']:.2f}",
            })
        except Exception as exc:
            status_rows.append({
                "domain": domain,
                "model_name": model_path.stem,
                "rmse": "N/A",
                "r2": "N/A",
                "n_test": "N/A",
                "status": "blocked",
                "notes": str(exc),
            })
    return pd.DataFrame(status_rows)


out_df = build_status_rows()
out_path = TABLE_ROOT / "external_transfer_summary.csv"
out_df.to_csv(out_path, index=False)

report_lines = [
    "# Notebook 15 External Transfer Status",
    "",
    f"External data files discovered: {len(discover_external_sources())}",
    f"Model artifacts discovered: {len(discover_model_artifacts())}",
    f"Summary artifact: {out_path}",
    "",
    "## Status by domain",
]
for _, row in out_df.iterrows():
    report_lines.append(
        f"- {row['domain']}: status={row['status']}, rmse={row['rmse']}, r2={row['r2']}, n_test={row['n_test']}, notes={row['notes']}"
    )

(REPORT_ROOT / "external_transfer_blocker_notes.md").write_text("\n".join(report_lines), encoding="utf-8")

print("Saved:", out_path)
print("Saved:", REPORT_ROOT / "external_transfer_blocker_notes.md")
out_df


Saved: c:\Users\obalo\Downloads\PhDOneDrive\PhD_Research_Operating_System\github_org_bootstrap\phd-geothermal-ml\manual_bootstrap\step_by_step_notebooks\15_external_transfer_no_scripts\outputs\summary\tables\external_transfer_summary.csv
Saved: c:\Users\obalo\Downloads\PhDOneDrive\PhD_Research_Operating_System\github_org_bootstrap\phd-geothermal-ml\manual_bootstrap\step_by_step_notebooks\15_external_transfer_no_scripts\outputs\summary\reports\external_transfer_blocker_notes.md


,domain,model_name,rmse,r2,n_test,status,notes
0,turkey_western_anatolia,gradient_boosting,N/A,N/A,N/A,blocked,external dataset not found in current reposito...
1,morocco_northern,gradient_boosting,N/A,N/A,N/A,blocked,external dataset not found in current reposito...
2,new_zealand_waikato,gradient_boosting,N/A,N/A,N/A,blocked,external dataset not found in current reposito...
